### Image Classification and Feature extraction — Eye: Male vs Female and Flower: 5 class
> Feature extraction and dimensionality reduction using pretrained models.


In [ ]:
# ─────────────────────────────────────────────
#  Block 1 · Imports & Path Configuration
# ─────────────────────────────────────────────

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import (
    VGG16, VGG19, ResNet50,
    MobileNet, MobileNetV2, DenseNet121,
    InceptionV3, Xception, EfficientNetB0,
)
from tensorflow.keras.applications.imagenet_utils import decode_predictions

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

!pip install -q umap-learn
import umap


# ── Paths ──────────────────────────────────────
BASE_DIR = "/content/drive/MyDrive/4-2/Deep Learning/Assignments/Assignment-1/Dataset"

DATASET_DIRS = {
    "flower": os.path.join(BASE_DIR, "flower"),
    "face":   os.path.join(BASE_DIR, "face"),
}

OUTPUT_DIR = (
    "/content/drive/MyDrive/4-2/Deep Learning/Assignments/"
    "Assignment-1/results/eye_classification_result"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")

In [ ]:
# ─────────────────────────────────────────────
#  Block 2 · Model Registry
# ─────────────────────────────────────────────

def fetch_model_config(model_name: str):
    """
    Return (model_class, preprocess_fn, image_size) for the given model name.

    Parameters
    ----------
    model_name : str
        One of: VGG16, VGG19, ResNet50, MobileNet, MobileNetV2,
        DenseNet121, InceptionV3, Xception, EfficientNetB0.

    Returns
    -------
    tuple  (model_class, preprocess_fn, (height, width))
    """
    registry = {
        "VGG16":         (VGG16,         tf.keras.applications.vgg16.preprocess_input,        (224, 224)),
        "VGG19":         (VGG19,         tf.keras.applications.vgg19.preprocess_input,        (224, 224)),
        "ResNet50":      (ResNet50,      tf.keras.applications.resnet50.preprocess_input,     (224, 224)),
        "MobileNet":     (MobileNet,     tf.keras.applications.mobilenet.preprocess_input,    (224, 224)),
        "MobileNetV2":   (MobileNetV2,   tf.keras.applications.mobilenet_v2.preprocess_input, (224, 224)),
        "DenseNet121":   (DenseNet121,   tf.keras.applications.densenet.preprocess_input,     (224, 224)),
        "InceptionV3":   (InceptionV3,   tf.keras.applications.inception_v3.preprocess_input, (299, 299)),
        "Xception":      (Xception,      tf.keras.applications.xception.preprocess_input,     (299, 299)),
        "EfficientNetB0":(EfficientNetB0,tf.keras.applications.efficientnet.preprocess_input, (224, 224)),
    }

    if model_name not in registry:
        raise ValueError(f"Unknown model '{model_name}'. Choose from: {list(registry)}")

    return registry[model_name]

In [ ]:
# ─────────────────────────────────────────────
#  Block 3 · Dataset Loader
# ─────────────────────────────────────────────

SUPPORTED_EXTENSIONS = (".jpg", ".jpeg", ".png")


def collect_image_paths(folder_path: str, dataset_name: str):
    """
    Walk a dataset folder and return parallel lists of image paths and labels.

    Labelling strategy
    ------------------
    - ``flower`` dataset : flat folder, label = filename prefix before the
      first dash  (e.g. ``rose-1.jpg`` → ``rose``).
    - ``face`` dataset   : one sub-folder per class; label = sub-folder name
      (e.g. ``male/``, ``female/``).

    Parameters
    ----------
    folder_path  : str   Absolute path to the dataset root.
    dataset_name : str   Either ``"flower"`` or ``"face"``.

    Returns
    -------
    image_paths : list[str]
    labels      : list[str]
    """
    image_paths, labels = [], []

    if dataset_name == "flower":
        # Flat directory — derive class from filename prefix
        for fname in os.listdir(folder_path):
            if fname.lower().endswith(SUPPORTED_EXTENSIONS):
                image_paths.append(os.path.join(folder_path, fname))
                labels.append(fname.split("-")[0])   # e.g. class1-1 → class1

    else:
        # Sub-folder per class — derive class from directory name
        for class_name in os.listdir(folder_path):
            class_dir = os.path.join(folder_path, class_name)
            if not os.path.isdir(class_dir):
                continue
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(SUPPORTED_EXTENSIONS):
                    image_paths.append(os.path.join(class_dir, fname))
                    labels.append(class_name)         # male / female

    return image_paths, labels

In [ ]:
# ─────────────────────────────────────────────
#  Block 4 · Single-Image Preprocessor
# ─────────────────────────────────────────────

def preprocess_single_image(img_path: str, preprocess_fn, img_size: tuple):
    """
    Load one image from disk, resize it, and apply the model-specific
    preprocessing function.

    Returns
    -------
    tensor : np.ndarray  Shape (1, H, W, 3) — ready for model.predict().
    pil_img              Original PIL Image (for display).
    """
    pil_img = image.load_img(img_path, target_size=img_size)
    arr     = image.img_to_array(pil_img)
    arr     = np.expand_dims(arr, axis=0)
    tensor  = preprocess_fn(arr)
    return tensor, pil_img

In [ ]:
# ─────────────────────────────────────────────
#  Block 5 · Classification & Visualisation
# ─────────────────────────────────────────────

NUM_PREVIEW_IMAGES = 5
TOP_K_PREDICTIONS  = 5


def classify_and_save_preview(
    model,
    preprocess_fn,
    model_name:  str,
    image_paths: list,
    labels:      list,
    img_size:    tuple,
):
    """
    Run ImageNet classification on up to ``NUM_PREVIEW_IMAGES`` samples,
    print top-K predictions, and save a multi-panel figure.

    Saved to: ``<OUTPUT_DIR>/<model_name>_result.png``
    """
    n_preview = min(NUM_PREVIEW_IMAGES, len(image_paths))
    fig, axes = plt.subplots(1, n_preview, figsize=(20, 5))

    for idx in range(n_preview):
        img_path   = image_paths[idx]
        true_label = labels[idx]

        tensor, pil_img = preprocess_single_image(img_path, preprocess_fn, img_size)

        preds   = model.predict(tensor, verbose=0)
        decoded = decode_predictions(preds, top=TOP_K_PREDICTIONS)[0]
        top1    = decoded[0][1]

        axes[idx].imshow(pil_img)
        axes[idx].axis("off")
        axes[idx].set_title(f"True: {true_label}\nPred: {top1}")

        print(f"\n[{model_name}] {os.path.basename(img_path)}")
        for rank, pred in enumerate(decoded, start=1):
            print(f"  Top-{rank}: {pred[1]} ({pred[2]:.4f})")

    save_path = os.path.join(OUTPUT_DIR, f"{model_name}_result.png")
    plt.savefig(save_path)
    plt.close()
    print(f"\nSaved classification preview → {save_path}")

In [ ]:
# ─────────────────────────────────────────────
#  Block 6 · Deep Feature Extractor
# ─────────────────────────────────────────────

def extract_deep_features(
    feature_model,
    preprocess_fn,
    image_paths: list,
    labels:      list,
    img_size:    tuple,
):
    """
    Pass every image through ``feature_model`` (top removed, global avg pool)
    and return a stacked feature matrix alongside the corresponding labels.

    Returns
    -------
    features : np.ndarray  Shape (N, D)
    labels   : np.ndarray  Shape (N,)
    """
    feature_list = []
    label_list   = []

    for idx, path in enumerate(image_paths):
        tensor, _ = preprocess_single_image(path, preprocess_fn, img_size)
        feat      = feature_model.predict(tensor, verbose=0)
        feature_list.append(feat)
        label_list.append(labels[idx])

    features = np.vstack(feature_list)
    labels   = np.array(label_list)

    print(f"Feature matrix shape : {features.shape}")
    print(f"Unique classes found : {np.unique(labels)}")

    return features, labels

In [ ]:
# ─────────────────────────────────────────────
#  Block 7 · Dimensionality Reduction
# ─────────────────────────────────────────────

def apply_dimensionality_reduction(features: np.ndarray):
    """
    Reduce a high-dimensional feature matrix to 2-D using three methods:
    PCA, t-SNE, and UMAP.

    Parameters
    ----------
    features : np.ndarray  Shape (N, D)

    Returns
    -------
    pca_embedding  : np.ndarray  (N, 2)
    tsne_embedding : np.ndarray  (N, 2)
    umap_embedding : np.ndarray  (N, 2)
    """
    print("Running PCA...")
    pca_embedding = PCA(n_components=2).fit_transform(features)

    print("Running t-SNE...")
    tsne_embedding = TSNE(n_components=2, perplexity=5, random_state=42).fit_transform(features)

    print("Running UMAP...")
    umap_embedding = umap.UMAP().fit_transform(features)

    return pca_embedding, tsne_embedding, umap_embedding

In [ ]:
# ─────────────────────────────────────────────
#  Block 8 · Embedding Scatter Plot
# ─────────────────────────────────────────────

def save_embedding_plot(
    embeddings: np.ndarray,
    labels:     np.ndarray,
    title:      str,
    filename:   str,
):
    """
    Draw a 2-D scatter plot colour-coded by class label and save to disk.

    Saved to: ``<OUTPUT_DIR>/<filename>``
    """
    plt.figure(figsize=(6, 5))

    for class_label in np.unique(labels):
        mask = labels == class_label
        plt.scatter(
            embeddings[mask, 0],
            embeddings[mask, 1],
            label=class_label,
        )

    plt.legend()
    plt.title(title)

    save_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(save_path)
    plt.close()
    print(f"Saved embedding plot → {save_path}")

In [ ]:
# ─────────────────────────────────────────────
#  Block 9 · Main Pipeline
# ─────────────────────────────────────────────

def execute_pipeline(model_name: str, dataset_name: str = "flower"):
    """
    End-to-end pipeline for a single model + dataset combination:

    1. Load pretrained model.
    2. Collect image paths and ground-truth labels.
    3. Run ImageNet classification and save a preview figure.
    4. Extract deep features (no top, global avg pool).
    5. Reduce to 2-D with PCA, t-SNE, and UMAP.
    6. Save one scatter plot per reduction method.

    Parameters
    ----------
    model_name   : str  Key from ``fetch_model_config`` registry.
    dataset_name : str  Either ``"flower"`` or ``"face"``.
    """
    print(f"\n{'='*40}")
    print(f"  Model   : {model_name}")
    print(f"  Dataset : {dataset_name}")
    print(f"{'='*40}\n")

    # ── Configuration ──────────────────────────
    model_class, preprocess_fn, img_size = fetch_model_config(model_name)

    # ── Dataset ────────────────────────────────
    dataset_dir = DATASET_DIRS[dataset_name]
    image_paths, labels = collect_image_paths(dataset_dir, dataset_name)
    print(f"Images found: {len(image_paths)}")

    # ── Classification ─────────────────────────
    clf_model = model_class(weights="imagenet")
    classify_and_save_preview(
        clf_model, preprocess_fn, model_name,
        image_paths, labels, img_size,
    )

    # ── Feature Extraction ─────────────────────
    feat_model = model_class(weights="imagenet", include_top=False, pooling="avg")
    features, feat_labels = extract_deep_features(
        feat_model, preprocess_fn, image_paths, labels, img_size,
    )

    # ── Dimensionality Reduction ───────────────
    pca_emb, tsne_emb, umap_emb = apply_dimensionality_reduction(features)

    # ── Save Plots ─────────────────────────────
    save_embedding_plot(pca_emb,  feat_labels, f"{model_name} — PCA",   f"{model_name}_pca.png")
    save_embedding_plot(tsne_emb, feat_labels, f"{model_name} — t-SNE", f"{model_name}_tsne.png")
    save_embedding_plot(umap_emb, feat_labels, f"{model_name} — UMAP",  f"{model_name}_umap.png")

In [ ]:
# ─────────────────────────────────────────────
#  Block 10 · Quick Test (single model)
# ─────────────────────────────────────────────

execute_pipeline("ResNet50", "face")

In [ ]:
# ─────────────────────────────────────────────
#  Block 11 · Full Benchmark (all models)
# ─────────────────────────────────────────────

ALL_MODELS = [
    "VGG16",
    "VGG19",
    "ResNet50",
    "MobileNet",
    "MobileNetV2",
    "DenseNet121",
    "InceptionV3",
    "Xception",
    "EfficientNetB0",
]


def benchmark_all_models(dataset_name: str = "flower"):
    """Run ``execute_pipeline`` for every model in ``ALL_MODELS``."""
    for model_name in ALL_MODELS:
        try:
            execute_pipeline(model_name, dataset_name)
        except Exception as err:
            print(f"[ERROR] {model_name}: {err}")


# benchmark_all_models()